# 回归像素空间

在过去广大的时间内，直接建立在像素空间上的模型训练或推理被认为面临巨大计算量的困难。因此，早期的 Autoregressive Models 与 像素空间上 DDPM 被遗弃，取而代之的则是利用 VAE 压缩像素空间到低维潜空间的 LDM 架构。原因是著名的流形假设认为真实语义嵌入在低维流形上，真实的高维空间非常空旷。

然而，受制于 VAE 困难的训练与性能瓶颈，近期我们又开始重新思考如何回归像素空间，以获取更加高质量的图像生成效果。

推荐你读 https://arxiv.org/abs/2511.20645 PixelDiT: Pixel Diffusion Transformers for Image Generation 成果来自 NVIDIA。这是本教程目前为止最前沿的成果介绍。代码仓库 https://github.com/NVlabs/PixelDiT PixelDiT 在 ImageNet 数据集上超越了 LDM 框架的同量级模型。

# 基本逻辑

PixelDiT 提出了一种高效处理图像像素空间的神经网络架构，核心的图像生成训练与推理算法仍是 Flow Matching，只是学习的概率分布从 LDM 框架下的潜空间概率分布变为像素空间的概率分布。下面这张图详细展示了 PixelDiT 的整体架构。

<img src="./assets/PixelDiT.png" width="1000" height="300">

我们回顾之前提到的 DiT 原版内部架构。

<img src="./assets/DiT.png" width="1000" height="480">

请忽视 DiT 架构图提到的潜空间相关内容，我们现在在真实高维的像素空间上原生地处理一切张量。

为你解释 PixelDiT 的架构。我们分两部分处理语义信息。首先是大尺度的 Patchify 操作，将原始 $(H, W, C)$ 张量沿着 $H \times W$ 平面分割成 $P \times P$ 小块，图像被切成 $L = \frac{H}{P} \times \frac{W}{P}$ 个块，最终每块展平为一维向量后的张量形状是 $(L, P^2 \cdot C)$。换句话说，我们将空间维度信息转换到了通道维度。

通过一个权重矩阵 $W \in \mathbb{R}^{(P^2 \cdot C) \times D}$ 将每个块映射到模型的隐藏维度 $D$。最终结果的张量维度是 $(L, D)$。

如果我们加上 Batch 维度，实际上是四维张量 $(B, C, H, W)$ 在 Patchify 之后变形为三维张量 $(B, L, D)$。这与我们在讲述 DiT 时说的一模一样。

我们不赘述条件与时间步的嵌入，这些内容在我们讲述 DiT 时就已经很明确，包括 AdaLN-Zero 调制技术。不过作者做了一个简单改进是将 DiT 的 LayerNorm 改为 RMSNorm，同时将位置编码升级为 RoPE 旋转位置编码。总之，经过 $N \times$ 个 DiT Blocks 之后，原始的像素数据张量 $(B, C, H, W)$ 变为了含有多层语义的 $(B, L, D)$ Semantic 张量。

现在我们来详谈 PiT Blocks，这是 PixelDiT 的核心创新。

我们首先接收来自 Semantic 部分的张量 $(B, L, D)$，为其加上时间步 $t$ 的编码 $(B, 1, D)$ 并且展开序列维度 $(BL, D)$ 作为 PiT 部分的条件编码。

其次的，来自原始像素数据的张量 $(B, C, H, W)$ 经历一个简单的变换变为 $(BL, P^2, C)$，进入 PiT Blocks。

我们先处理条件位置编码。一个线性投影层 $\text{Linear}(D, 6 * P^2 * C)$ 将其变为 $(BL, 6 * P^2 * C)$，然后对通道维度切分 $6$ 次并且升维。这 $6$ 个参数就是 adaLN-Zero 的 $3$ 组缩放与偏移参数，形状是 $(BL,P^2,C)$。这是条件部分的全部处理。

请注意 DiT 原版的 adaLN-Zero 是选择生成一个 $(BL,1 ,C)$ 形状的张量做广播，但是 PixelDiT 选择了完整地为每个位置赋予放缩与偏移参数，这被称为 Pixel-wise AdaLN 调制。因此处理 Semantic 张量的 DiT Blocks 中使用的 adaLN 与 PiT 中的并不完全相同。

回到架构图中下方的像素张量 $(BL, P^2, C)$ 输入。其会先经历 RMSNorm，之后应用 adaLN-Zero 所产生的放缩与偏移参数。在进入 Multi-Head Self-Attention 之前，为了减少计算量，我们先做一次线性层压缩。展平张量为 $(BL, P^2 * C)$ 经过压缩线性层 $\text{Linear}(P^2 * C, A)$ 得到 $(BL,A)$。其中 $A$ 是注意力层隐藏维度。

升维到 $(B,L, A)$ 做多头自注意力运算，这一步保持形状。然后再次将 Batch 维度与序列维度合并并且投影回到 $(BL, P^2 * C)$，并且为了残差连接还会再升维到 $(BL, P^2 ,C)$。

之后的模块操作是常规的。最终 PiT Blocks 会输出张量 $(BL, P^2 ,C)$，做变换得到符合真实像素空间元素形状的 $(B, C, H, W)$ 作为整个模型的输出。

所以 PixelDiT 整个过程其实取巧颇多。最重要的就是 PiT 中做多头自注意力之前的线性层投影压缩从而减少计算量。

需要注意，以上是原论文中 Class2Image 的模型架构，类似输入一个 Label 做图像生成。原作者还提到了他们的 Text2Image，也就是更符合实际工业生成的根据文字生成图片。两者总的架构没有巨大区别，只是 T2I 在对于条件嵌入的处理会更细腻，使用了我们在 SD3 章节中提到的 MM-DiT Blocks 与专门的 Gemma2 文字编码器。我们不赘述这部分内容。

值得一提，PixelDiT 的一个重要突破就是推理速度。在 $512 \times 512$ 分辨率上的图像生成任务中 PixelDiT-T2I 的吞吐量为 $1.07$ 张每秒，持平甚至超越目前主流的潜空间扩散模型。这对比过去的像素级生成模型是一个巨大进展，揭露了像素级生成模型进入工业生产的可能性。更多的，PixelDiT 如预料地在像素空间上展现出了不俗的生成质量，如强大的细节文字处理能力。

# 总结

本章介绍了图像生成领域目前对于重回像素空间生成的思考。在计算技术与硬件逐渐升级的大背景下，我们开始将图像生成质量放置到更高的位置。

实不相瞒，本章如此短的原因是原本还想要介绍来自何恺明团队的重要成果 Joint Image-Text Diffusion，同样是对于回归像素空间的尝试。他们证明了像素级生成模型是可以放大参数量提升生成质量的，这为未来的像素级生成模型建立了理论基础。但是我突然不愿意继续在这里停留了。

本章是图像生成主线的最后一章，之后的章节会以专题形式讲解各个重要技术。我将第一个专题主题定为加速生成。尽管我们其实已经有所涉猎，但是我愿意介绍更多先进的技术方法与细节。